<a href="https://colab.research.google.com/github/beckysianga/springerprojects.github.io/blob/main/SpringerDiabetesDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# PR-AS-HDP with RDP Accounting and Ablation Study
# Diabetes Dataset
# ============================================================
# Ablations:
#   A1 = ClipOnly
#   A2 = Clip + Adaptive Budget
#   A3 = Clip + Adaptive Budget + Client Sensitivity
#   A4 = Full PR-AS-HDP + RDP (local Gaussian + central Gaussian)
#
# Main additions:
# - RDP accountant for local and central Gaussian mechanisms
# - Full ablation framework
# - Multi-seed evaluation
# - Per-round privacy logging
# - Personalised FL with shared encoder + local heads
# ============================================================

import copy
import json
import math
import os
import random
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# ============================================================
# 1. Utilities
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def safe_np(x: np.ndarray) -> np.ndarray:
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)


def safe_tensor(x: torch.Tensor, clip_val: float = 20.0) -> torch.Tensor:
    return torch.nan_to_num(x, nan=0.0, posinf=clip_val, neginf=-clip_val)


# ============================================================
# 2. Configuration
# ============================================================

@dataclass
class Config:
    # Data
    csv_path: str = "/content/diabetes.csv"
    target_col: str = "Diabetes_binary"
    positive_label: int = 1

    # Split
    test_size: float = 0.10
    val_size: float = 0.10

    # FL
    n_clients: int = 8
    rounds: int = 30
    client_frac: float = 0.75
    local_epochs: int = 2
    batch_size: int = 128
    lr_encoder: float = 4e-4
    lr_head: float = 7e-4
    weight_decay: float = 1e-5
    dirichlet_alpha: float = 0.5

    # Model
    hidden_dim1: int = 128
    hidden_dim2: int = 64
    rep_dim: int = 32
    head_hidden: int = 32
    dropout: float = 0.10
    use_batchnorm: bool = True

    # Personalisation
    personal_encoder_blend: float = 0.20

    # Adaptive sensitivity
    lambda1: float = 0.20     # representation dispersion
    lambda2: float = 0.20     # clipped update magnitude
    lambda3: float = 0.05     # imbalance factor
    sensitivity_cap: float = 250.0
    sensitivity_floor: float = 1e-6

    # Clipping
    alpha_ema: float = 0.90
    quantile_q: float = 0.80
    cmin: float = 0.50
    cmax: float = 6.00
    initial_r: float = 1.60

    # Budget schedule
    eps_base_local: float = 0.80
    eps_base_central: float = 0.60
    eps_min: float = 0.15
    eps_max: float = 2.50
    beta_sched: float = 0.04
    gamma_imbalance: float = 0.25
    warmup_rounds: int = 8
    fixed_warmup_eps: float = 0.90

    # DP
    delta: float = 1e-5
    rdp_orders: Tuple[float, ...] = (
        1.25, 1.5, 2, 3, 4, 5, 8, 10, 16, 32, 64, 128
    )
    max_noise_scale: float = 10.0

    # Loss
    focal_gamma: float = 2.0
    focal_alpha: float = 0.75

    # Early stopping
    patience: int = 8
    min_delta: float = 1e-4

    # Experiment
    seeds: Tuple[int, ...] = (42, 52, 62)
    output_dir: str = "/content/pr_ashdp_rdp_outputs"


CFG = Config()

# ============================================================
# 3. Data Loading and Partitioning
# ============================================================

def load_diabetes_csv(csv_path: str, target_col: str, positive_label: int = 1):
    df = pd.read_csv(csv_path)

    if target_col not in df.columns:
        raise ValueError(
            f"Target column '{target_col}' not found. Available columns: {list(df.columns)}"
        )

    y = df[target_col].copy()
    X = df.drop(columns=[target_col]).copy()

    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    y = pd.to_numeric(y, errors="coerce")

    X = X.replace([np.inf, -np.inf], np.nan)
    y = y.replace([np.inf, -np.inf], np.nan)

    X = X.fillna(X.median(numeric_only=True))
    X = X.fillna(0)
    y = y.fillna(0)

    X = X.to_numpy(dtype=np.float32)
    y = (y.to_numpy() == positive_label).astype(np.float32)

    X = safe_np(X).astype(np.float32)
    y = safe_np(y).astype(np.float32)

    return X, y


def split_data(X: np.ndarray, y: np.ndarray, cfg: Config, seed: int):
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y,
        test_size=cfg.test_size,
        random_state=seed,
        stratify=y
    )

    val_ratio = cfg.val_size / (1.0 - cfg.test_size)

    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=val_ratio,
        random_state=seed,
        stratify=y_trainval
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    return (
        safe_np(X_train),
        safe_np(y_train),
        safe_np(X_val),
        safe_np(y_val),
        safe_np(X_test),
        safe_np(y_test),
        scaler,
    )


def dirichlet_partition(X: np.ndarray, y: np.ndarray, n_clients: int, alpha: float, seed: int):
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    class_indices = [np.where(y == c)[0] for c in classes]
    client_indices = [[] for _ in range(n_clients)]

    for idx in class_indices:
        idx = idx.copy()
        rng.shuffle(idx)
        proportions = rng.dirichlet([alpha] * n_clients)
        cuts = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
        splits = np.split(idx, cuts)
        for cid, sp in enumerate(splits):
            client_indices[cid].extend(sp.tolist())

    clients = []
    for cid, idxs in enumerate(client_indices):
        idxs = np.array(idxs, dtype=int)
        rng.shuffle(idxs)
        Xc = safe_np(X[idxs]).astype(np.float32)
        yc = safe_np(y[idxs]).astype(np.float32)
        clients.append((Xc, yc))
    return clients


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool = True):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


# ============================================================
# 4. Model
# ============================================================

class SharedEncoder(nn.Module):
    def __init__(self, input_dim: int, cfg: Config):
        super().__init__()
        layers = [nn.Linear(input_dim, cfg.hidden_dim1)]
        if cfg.use_batchnorm:
            layers.append(nn.BatchNorm1d(cfg.hidden_dim1))
        layers += [nn.ReLU(), nn.Dropout(cfg.dropout)]

        layers += [nn.Linear(cfg.hidden_dim1, cfg.hidden_dim2)]
        if cfg.use_batchnorm:
            layers.append(nn.BatchNorm1d(cfg.hidden_dim2))
        layers += [nn.ReLU(), nn.Dropout(cfg.dropout)]

        layers += [nn.Linear(cfg.hidden_dim2, cfg.rep_dim), nn.ReLU()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class PersonalHead(nn.Module):
    def __init__(self, rep_dim: int, head_hidden: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rep_dim, head_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1)
        )

    def forward(self, z):
        return self.net(z)


class PersonalisedClientModel(nn.Module):
    def __init__(self, encoder: SharedEncoder, head: PersonalHead):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, x):
        z = self.encoder(x)
        logits = self.head(z)
        return logits, z


class InferenceModel(nn.Module):
    def __init__(self, encoder: nn.Module, head: nn.Module):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, x):
        z = self.encoder(x)
        logits = self.head(z)
        return logits, z


def make_inference_model(encoder: nn.Module, head: nn.Module):
    return InferenceModel(copy.deepcopy(encoder), copy.deepcopy(head)).to(DEVICE)


# ============================================================
# 5. Loss
# ============================================================

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        logits = safe_tensor(logits)
        targets = targets.float()

        bce = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )

        probs = torch.sigmoid(logits)
        probs = torch.clamp(probs, min=1e-6, max=1 - 1e-6)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        focal = alpha_t * ((1 - pt) ** self.gamma) * bce

        if self.reduction == "mean":
            return focal.mean()
        if self.reduction == "sum":
            return focal.sum()
        return focal


# ============================================================
# 6. Metrics
# ============================================================

@torch.no_grad()
def predict_probs_model(model: nn.Module, X: np.ndarray) -> np.ndarray:
    model.eval()
    xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    logits, _ = model(xt)
    logits = safe_tensor(logits)
    probs = torch.sigmoid(logits).squeeze().detach().cpu().numpy()
    return np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)


def find_best_threshold(y_true: np.ndarray, probs: np.ndarray):
    best_thr = 0.5
    best_f1 = -1.0
    for thr in np.linspace(0.10, 0.90, 81):
        preds = (probs >= thr).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thr = thr
    return best_thr, best_f1


def evaluate_probs(y: np.ndarray, probs: np.ndarray, threshold: float = 0.5):
    y = safe_np(y).astype(int)
    probs = safe_np(probs)
    preds = (probs >= threshold).astype(int)

    out = {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "auprc": average_precision_score(y, probs),
    }
    try:
        out["auroc"] = roc_auc_score(y, probs)
    except Exception:
        out["auroc"] = np.nan
    return out


# ============================================================
# 7. Parameter Helpers
# ============================================================

def flatten_params(module: nn.Module) -> torch.Tensor:
    return torch.cat([p.data.view(-1) for p in module.parameters()])


def set_params_from_flat(module: nn.Module, flat: torch.Tensor):
    ptr = 0
    for p in module.parameters():
        n = p.numel()
        p.data.copy_(flat[ptr:ptr + n].view_as(p))
        ptr += n


def get_update(local_module: nn.Module, global_module: nn.Module) -> torch.Tensor:
    return flatten_params(local_module) - flatten_params(global_module)


def l2norm(t: torch.Tensor) -> float:
    return torch.norm(t, p=2).item()


def blend_encoder(local_encoder: nn.Module, global_encoder: nn.Module, strength: float):
    with torch.no_grad():
        for p_l, p_g in zip(local_encoder.parameters(), global_encoder.parameters()):
            delta = p_l.data - p_g.data
            delta = safe_tensor(delta, clip_val=50.0)
            p_l.data = p_g.data + strength * delta
            p_l.data = safe_tensor(p_l.data, clip_val=50.0)


# ============================================================
# 8. Adaptive Mechanisms
# ============================================================

def class_proportions(y: np.ndarray) -> Dict[int, float]:
    total = max(len(y), 1)
    return {int(c): float((y == c).sum()) / total for c in np.unique(y)}


def imbalance_factor(y: np.ndarray, eps: float = 1e-8) -> float:
    props = class_proportions(y)
    if len(props) == 0:
        return 1.0
    k = max(len(props), 1)
    return sum(1.0 / (props[c] + eps) for c in props) / k


@torch.no_grad()
def representation_dispersion(encoder: nn.Module, X: np.ndarray) -> float:
    if len(X) == 0:
        return 0.0
    encoder.eval()
    xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    z = encoder(xt)
    z = safe_tensor(z, clip_val=50.0)
    std_vec = torch.std(z, dim=0)
    std_vec = safe_tensor(std_vec, clip_val=50.0)
    return torch.norm(std_vec, p=2).item()


def adaptive_clipping_threshold(prev_r: float, norms: List[float], cfg: Config):
    if len(norms) == 0:
        return prev_r, max(cfg.cmin, prev_r)

    norms = np.nan_to_num(np.array(norms), nan=0.0, posinf=0.0, neginf=0.0)
    qv = float(np.quantile(norms, cfg.quantile_q))
    r_t = cfg.alpha_ema * prev_r + (1.0 - cfg.alpha_ema) * qv
    c_t = float(np.clip(r_t, cfg.cmin, cfg.cmax))
    return r_t, c_t


def clip_update(update: torch.Tensor, c_t: float) -> torch.Tensor:
    update = safe_tensor(update, clip_val=50.0)
    norm = torch.norm(update, p=2)
    if norm.item() <= c_t:
        return update
    out = update * (c_t / (norm + 1e-12))
    return safe_tensor(out, clip_val=50.0)


def compute_sensitivity(
    encoder: nn.Module,
    X_client: np.ndarray,
    clipped_update: torch.Tensor,
    y_client: np.ndarray,
    cfg: Config
) -> float:
    rep_disp = representation_dispersion(encoder, X_client)
    upd_mag = l2norm(clipped_update)
    imb = imbalance_factor(y_client)
    sens = cfg.lambda1 * rep_disp + cfg.lambda2 * upd_mag + cfg.lambda3 * imb
    sens = float(np.nan_to_num(sens, nan=0.0, posinf=0.0, neginf=0.0))
    sens = max(sens, cfg.sensitivity_floor)
    sens = min(sens, cfg.sensitivity_cap)
    return sens


def budget_scheduler(round_idx: int, prev_val_auprc: float, current_val_auprc: float, cfg: Config):
    if round_idx <= cfg.warmup_rounds:
        return cfg.fixed_warmup_eps

    delta_auprc = max(current_val_auprc - prev_val_auprc, 1e-5)
    eps_t = cfg.eps_base_local * (1.0 + cfg.beta_sched * (1.0 / (delta_auprc + 1e-5)))
    return float(np.clip(eps_t, cfg.eps_min, cfg.eps_max))


def class_conditional_budgets(y_client: np.ndarray, eps_t: float, gamma: float):
    props = class_proportions(y_client)
    if len(props) == 0:
        return {0: eps_t}
    raw = {c: 1.0 / ((props[c] + 1e-8) ** gamma) for c in props}
    denom = max(sum(raw.values()), 1e-8)
    return {c: eps_t * raw[c] / denom for c in raw}


# ============================================================
# 9. RDP Accountant
# ============================================================

class RDPAccountant:
    """
    Simple RDP accountant for Gaussian mechanisms.

    For Gaussian mechanism with sensitivity 1 and noise std sigma:
        RDP(alpha) = alpha / (2 * sigma^2)

    For general L2 sensitivity S and noise std std:
        sigma_equiv = std / S
        RDP(alpha) = alpha / (2 * sigma_equiv^2)
                   = alpha * S^2 / (2 * std^2)
    """

    def __init__(self, orders: Tuple[float, ...], delta: float):
        self.orders = list(orders)
        self.delta = delta
        self.rdp = {a: 0.0 for a in self.orders}

    def add_gaussian_mechanism(self, sensitivity: float, std: float):
        sensitivity = float(max(sensitivity, 1e-12))
        std = float(max(std, 1e-12))
        for a in self.orders:
            self.rdp[a] += (a * (sensitivity ** 2)) / (2.0 * (std ** 2))

    def get_epsilon(self):
        eps_candidates = []
        for a in self.orders:
            if a <= 1:
                continue
            eps = self.rdp[a] + math.log(1.0 / self.delta) / (a - 1.0)
            eps_candidates.append((a, eps))
        best_alpha, best_eps = min(eps_candidates, key=lambda x: x[1])
        return float(best_eps), float(best_alpha)

    def snapshot(self):
        eps, alpha = self.get_epsilon()
        snap = {"epsilon": eps, "best_alpha": alpha}
        for a in self.orders:
            snap[f"rdp_alpha_{str(a).replace('.', '_')}"] = self.rdp[a]
        return snap


def gaussian_std_from_eps(sensitivity: float, eps: float, delta: float):
    sensitivity = max(float(sensitivity), 1e-12)
    eps = max(float(eps), 1e-12)
    return sensitivity * math.sqrt(2.0 * math.log(1.25 / delta)) / eps


def gaussian_noise_like(t: torch.Tensor, std: float, cfg: Config):
    std = float(np.clip(std, 0.0, cfg.max_noise_scale))
    return safe_tensor(torch.randn_like(t) * std, clip_val=50.0)


# ============================================================
# 10. Local Training
# ============================================================

def train_local_personalised(
    global_encoder: nn.Module,
    personal_head: nn.Module,
    Xc: np.ndarray,
    yc: np.ndarray,
    cfg: Config,
):
    local_encoder = copy.deepcopy(global_encoder).to(DEVICE)
    local_head = copy.deepcopy(personal_head).to(DEVICE)

    model = PersonalisedClientModel(local_encoder, local_head).to(DEVICE)
    loader = make_loader(Xc, yc, batch_size=cfg.batch_size, shuffle=True)
    criterion = BinaryFocalLoss(alpha=cfg.focal_alpha, gamma=cfg.focal_gamma, reduction="mean")

    optimizer = torch.optim.Adam(
        [
            {"params": model.encoder.parameters(), "lr": cfg.lr_encoder},
            {"params": model.head.parameters(), "lr": cfg.lr_head},
        ],
        weight_decay=cfg.weight_decay,
    )

    epoch_losses = []
    model.train()

    for _ in range(cfg.local_epochs):
        batch_losses = []
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).unsqueeze(1)

            optimizer.zero_grad()
            logits, _ = model(xb)
            logits = safe_tensor(logits)
            loss = criterion(logits, yb)

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            loss.backward()

            for p in model.parameters():
                if p.grad is not None:
                    p.grad.data = safe_tensor(p.grad.data, clip_val=10.0)

            optimizer.step()

            for p in model.parameters():
                p.data = safe_tensor(p.data, clip_val=50.0)

            batch_losses.append(float(loss.item()))

        epoch_losses.append(float(np.mean(batch_losses)) if batch_losses else np.nan)

    blend_encoder(model.encoder, global_encoder, strength=cfg.personal_encoder_blend)

    encoder_update = get_update(model.encoder, global_encoder)
    encoder_update = safe_tensor(encoder_update, clip_val=50.0)

    track = {
        "local_loss_mean": float(np.nanmean(epoch_losses)) if len(epoch_losses) > 0 else np.nan,
        "local_loss_last": float(epoch_losses[-1]) if len(epoch_losses) > 0 else np.nan,
    }

    return model.encoder, model.head, encoder_update, track


# ============================================================
# 11. Evaluation
# ============================================================

def evaluate_global_model(global_encoder: nn.Module, global_head: nn.Module, X: np.ndarray, y: np.ndarray):
    model = make_inference_model(global_encoder, global_head)
    probs = predict_probs_model(model, X)
    thr, _ = find_best_threshold(y, probs)
    metrics = evaluate_probs(y, probs, threshold=thr)
    return metrics, thr, probs


def evaluate_personalised_clients(
    global_encoder: nn.Module,
    client_heads: Dict[int, nn.Module],
    clients: List[Tuple[np.ndarray, np.ndarray]],
):
    rows = []
    all_probs = []
    all_y = []

    for cid, (Xc, yc) in enumerate(clients):
        if len(Xc) == 0:
            continue
        head = client_heads[cid]
        model = make_inference_model(global_encoder, head)
        probs = predict_probs_model(model, Xc)
        thr, _ = find_best_threshold(yc, probs)
        metrics = evaluate_probs(yc, probs, threshold=thr)
        metrics["client_id"] = cid
        metrics["threshold"] = thr
        metrics["n_samples"] = len(Xc)
        rows.append(metrics)
        all_probs.append(probs)
        all_y.append(yc)

    client_df = pd.DataFrame(rows)

    if len(all_probs) > 0:
        pooled_probs = np.concatenate(all_probs)
        pooled_y = np.concatenate(all_y)
        pooled_thr, _ = find_best_threshold(pooled_y, pooled_probs)
        pooled_metrics = evaluate_probs(pooled_y, pooled_probs, threshold=pooled_thr)
        pooled_metrics["threshold"] = pooled_thr
    else:
        pooled_metrics = {
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "auprc": np.nan,
            "auroc": np.nan,
            "threshold": np.nan,
        }

    return client_df, pooled_metrics


# ============================================================
# 12. Ablation Logic
# ============================================================

ABLATION_LABELS = {
    "A1": "ClipOnly",
    "A2": "ClipBudget",
    "A3": "ClipBudgetClientSens",
    "A4": "FullPRASHDP_RDP",
}


def get_ablation_flags(ablation: str):
    ablation = ablation.upper()
    if ablation == "A1":
        return {
            "use_adaptive_budget": False,
            "use_client_sensitivity": False,
            "use_local_dp": False,
            "use_central_dp": False,
            "use_rdp": False,
        }
    if ablation == "A2":
        return {
            "use_adaptive_budget": True,
            "use_client_sensitivity": False,
            "use_local_dp": False,
            "use_central_dp": False,
            "use_rdp": False,
        }
    if ablation == "A3":
        return {
            "use_adaptive_budget": True,
            "use_client_sensitivity": True,
            "use_local_dp": False,
            "use_central_dp": False,
            "use_rdp": False,
        }
    if ablation == "A4":
        return {
            "use_adaptive_budget": True,
            "use_client_sensitivity": True,
            "use_local_dp": True,
            "use_central_dp": True,
            "use_rdp": True,
        }
    raise ValueError(f"Unknown ablation: {ablation}")


# ============================================================
# 13. Federated Training with RDP + Ablations
# ============================================================

def federated_train_pr_ashdp_personalised(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    cfg: Config,
    seed: int,
    ablation: str = "A4",
):
    seed_everything(seed)
    flags = get_ablation_flags(ablation)

    clients = dirichlet_partition(X_train, y_train, cfg.n_clients, cfg.dirichlet_alpha, seed=seed)

    input_dim = X_train.shape[1]
    global_encoder = SharedEncoder(input_dim, cfg).to(DEVICE)
    global_head = PersonalHead(cfg.rep_dim, cfg.head_hidden, cfg.dropout).to(DEVICE)

    client_heads = {
        cid: copy.deepcopy(global_head).to(DEVICE)
        for cid in range(cfg.n_clients)
    }

    prev_r = cfg.initial_r
    prev_val_auprc = 1e-5

    round_history = []
    client_round_history = []
    rdp_round_history = []

    accountant = RDPAccountant(cfg.rdp_orders, cfg.delta) if flags["use_rdp"] else None

    best_val_auprc = -np.inf
    best_encoder_state = None
    best_heads_state = None
    best_round = 0
    best_threshold = 0.5
    no_improve = 0

    for rnd in range(1, cfg.rounds + 1):
        val_global_metrics, _, _ = evaluate_global_model(global_encoder, global_head, X_val, y_val)
        current_val_auprc = val_global_metrics["auprc"]

        if flags["use_adaptive_budget"]:
            eps_t_local = budget_scheduler(rnd, prev_val_auprc, current_val_auprc, cfg)
            eps_t_central = float(np.clip(0.75 * eps_t_local, cfg.eps_min, cfg.eps_max))
        else:
            eps_t_local = cfg.fixed_warmup_eps
            eps_t_central = cfg.fixed_warmup_eps

        prev_val_auprc = current_val_auprc

        m = max(1, int(cfg.client_frac * cfg.n_clients))
        selected = np.random.choice(range(cfg.n_clients), size=m, replace=False)

        local_encoders = []
        raw_updates = []
        sizes = []
        client_ids = []
        sens_list = []
        clipped_updates = []

        for cid in selected:
            Xc, yc = clients[cid]
            if len(Xc) == 0:
                continue

            local_encoder, local_head, enc_update, track = train_local_personalised(
                global_encoder=global_encoder,
                personal_head=client_heads[cid],
                Xc=Xc,
                yc=yc,
                cfg=cfg,
            )

            client_heads[cid] = copy.deepcopy(local_head).to(DEVICE)

            local_encoders.append(local_encoder)
            raw_updates.append(enc_update)
            sizes.append(len(Xc))
            client_ids.append(cid)

            personal_model = make_inference_model(local_encoder, local_head)
            probs_client = predict_probs_model(personal_model, Xc)
            thr_client, _ = find_best_threshold(yc, probs_client)
            client_metrics = evaluate_probs(yc, probs_client, threshold=thr_client)

            client_row = {
                "round": rnd,
                "seed": seed,
                "ablation": ablation,
                "client_id": cid,
                "n_samples": len(Xc),
                "eps_t_local": eps_t_local,
                "eps_t_central": eps_t_central,
                "local_loss_mean": track["local_loss_mean"],
                "local_loss_last": track["local_loss_last"],
                "client_threshold": thr_client,
                "client_auprc": client_metrics["auprc"],
                "client_f1": client_metrics["f1"],
                "client_recall": client_metrics["recall"],
                "client_precision": client_metrics["precision"],
                "client_accuracy": client_metrics["accuracy"],
                "client_auroc": client_metrics["auroc"],
            }
            client_round_history.append(client_row)

        if len(raw_updates) == 0:
            print(f"Round {rnd}: no valid clients, skipping.")
            continue

        norms = [l2norm(u) for u in raw_updates]
        prev_r, c_t = adaptive_clipping_threshold(prev_r, norms, cfg)

        noisy_updates = []

        for cid, local_encoder, upd in zip(client_ids, local_encoders, raw_updates):
            Xc, yc = clients[cid]

            clipped = clip_update(upd, c_t)
            clipped_updates.append(clipped)

            if flags["use_client_sensitivity"]:
                sens = compute_sensitivity(local_encoder, Xc, clipped, yc, cfg)
            else:
                sens = max(c_t, cfg.sensitivity_floor)

            sens_list.append(sens)

            if flags["use_local_dp"]:
                budgets = class_conditional_budgets(yc, eps_t_local, cfg.gamma_imbalance)
                eps_ldp = float(np.mean(list(budgets.values()))) if len(budgets) > 0 else eps_t_local
                eps_ldp = max(eps_ldp, cfg.eps_min)
                std_local = gaussian_std_from_eps(sens, eps_ldp, cfg.delta)
                noisy = clipped + gaussian_noise_like(clipped.to(DEVICE), std_local, cfg)

                if accountant is not None:
                    accountant.add_gaussian_mechanism(sensitivity=sens, std=std_local)
            else:
                eps_ldp = np.nan
                std_local = 0.0
                noisy = clipped.clone()

            noisy = safe_tensor(noisy, clip_val=50.0)
            noisy_updates.append(noisy)

        agg_update = torch.zeros_like(flatten_params(global_encoder))
        total_size = max(sum(sizes), 1)

        for upd, sz in zip(noisy_updates, sizes):
            agg_update += (sz / total_size) * upd

        mean_sensitivity = float(np.mean(sens_list)) if len(sens_list) > 0 else cfg.sensitivity_floor

        if flags["use_central_dp"]:
            std_central = gaussian_std_from_eps(mean_sensitivity, eps_t_central, cfg.delta)
            agg_update = agg_update + gaussian_noise_like(agg_update, std_central, cfg)
            agg_update = safe_tensor(agg_update, clip_val=50.0)

            if accountant is not None:
                accountant.add_gaussian_mechanism(sensitivity=mean_sensitivity, std=std_central)
        else:
            std_central = 0.0

        new_flat = flatten_params(global_encoder) + agg_update
        new_flat = safe_tensor(new_flat, clip_val=50.0)
        set_params_from_flat(global_encoder, new_flat.to(DEVICE))

        head_state_dicts = [client_heads[cid].state_dict() for cid in client_heads]
        avg_head = copy.deepcopy(global_head).to(DEVICE)
        avg_state = avg_head.state_dict()
        for k in avg_state.keys():
            avg_state[k] = sum(h[k] for h in head_state_dicts) / len(head_state_dicts)
        avg_head.load_state_dict(avg_state)
        global_head = avg_head

        val_model = make_inference_model(global_encoder, global_head)
        val_probs = predict_probs_model(val_model, X_val)
        thr, _ = find_best_threshold(y_val, val_probs)
        val_metrics = evaluate_probs(y_val, val_probs, threshold=thr)

        test_model = make_inference_model(global_encoder, global_head)
        test_probs = predict_probs_model(test_model, X_test)
        test_metrics = evaluate_probs(y_test, test_probs, threshold=thr)

        _, pooled_client_metrics = evaluate_personalised_clients(
            global_encoder=global_encoder,
            client_heads=client_heads,
            clients=clients,
        )

        if accountant is not None:
            rdp_snap = accountant.snapshot()
            epsilon_total = rdp_snap["epsilon"]
            best_alpha = rdp_snap["best_alpha"]
        else:
            rdp_snap = {}
            epsilon_total = np.nan
            best_alpha = np.nan

        round_row = {
            "round": rnd,
            "seed": seed,
            "ablation": ablation,
            "ablation_name": ABLATION_LABELS[ablation],
            "eps_t_local": eps_t_local,
            "eps_t_central": eps_t_central,
            "clip_Ct": c_t,
            "mean_sensitivity": mean_sensitivity,
            "std_central": std_central,
            "val_best_threshold": thr,
            "val_auprc": val_metrics["auprc"],
            "val_f1": val_metrics["f1"],
            "test_auprc": test_metrics["auprc"],
            "test_f1": test_metrics["f1"],
            "test_recall": test_metrics["recall"],
            "test_precision": test_metrics["precision"],
            "test_accuracy": test_metrics["accuracy"],
            "test_auroc": test_metrics["auroc"],
            "personal_train_auprc": pooled_client_metrics["auprc"],
            "personal_train_f1": pooled_client_metrics["f1"],
            "personal_train_recall": pooled_client_metrics["recall"],
            "personal_train_precision": pooled_client_metrics["precision"],
            "rdp_epsilon_total": epsilon_total,
            "rdp_best_alpha": best_alpha,
        }
        round_history.append(round_row)

        if accountant is not None:
            rdp_row = {
                "round": rnd,
                "seed": seed,
                "ablation": ablation,
                **rdp_snap,
            }
            rdp_round_history.append(rdp_row)

        print(round_row)

        if val_metrics["auprc"] > best_val_auprc + cfg.min_delta:
            best_val_auprc = val_metrics["auprc"]
            best_encoder_state = copy.deepcopy(global_encoder.state_dict())
            best_heads_state = {cid: copy.deepcopy(client_heads[cid].state_dict()) for cid in client_heads}
            best_round = rnd
            best_threshold = thr
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= cfg.patience:
            print(f"\nEarly stopping at round {rnd}. Best round: {best_round}")
            break

    round_history_df = pd.DataFrame(round_history)
    client_round_history_df = pd.DataFrame(client_round_history)
    rdp_round_history_df = pd.DataFrame(rdp_round_history)

    if best_encoder_state is not None:
        global_encoder.load_state_dict(best_encoder_state)
        for cid in client_heads:
            client_heads[cid].load_state_dict(best_heads_state[cid])

    head_state_dicts = [client_heads[cid].state_dict() for cid in client_heads]
    best_global_head = PersonalHead(cfg.rep_dim, cfg.head_hidden, cfg.dropout).to(DEVICE)
    avg_state = best_global_head.state_dict()
    for k in avg_state.keys():
        avg_state[k] = sum(h[k] for h in head_state_dicts) / len(head_state_dicts)
    best_global_head.load_state_dict(avg_state)

    best_val_model = make_inference_model(global_encoder, best_global_head)
    final_val_probs = predict_probs_model(best_val_model, X_val)
    final_best_threshold, _ = find_best_threshold(y_val, final_val_probs)
    final_val_metrics = evaluate_probs(y_val, final_val_probs, threshold=final_best_threshold)

    best_test_model = make_inference_model(global_encoder, best_global_head)
    final_test_probs = predict_probs_model(best_test_model, X_test)
    final_test_metrics = evaluate_probs(y_test, final_test_probs, threshold=final_best_threshold)

    final_privacy = accountant.snapshot() if accountant is not None else {}

    print("\n================ FINAL BEST MODEL ================")
    print("Seed:", seed)
    print("Ablation:", ablation, ABLATION_LABELS[ablation])
    print("Best round:", best_round)
    print("Best threshold:", final_best_threshold)
    print("Validation metrics:", final_val_metrics)
    print("Test metrics:", final_test_metrics)
    print("Final privacy:", final_privacy)

    return {
        "global_encoder": global_encoder,
        "global_head": best_global_head,
        "client_heads": client_heads,
        "round_history": round_history_df,
        "client_round_history": client_round_history_df,
        "rdp_round_history": rdp_round_history_df,
        "final_val_metrics": final_val_metrics,
        "final_test_metrics": final_test_metrics,
        "final_privacy": final_privacy,
        "best_round": best_round,
        "best_threshold": final_best_threshold,
    }


# ============================================================
# 14. Multi-Seed Ablation Runner
# ============================================================

def summarize_metric(series: pd.Series):
    arr = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if len(arr) == 0:
        return np.nan, np.nan
    return float(np.mean(arr)), float(np.std(arr, ddof=0))


def run_all_ablation_experiments(cfg: Config):
    os.makedirs(cfg.output_dir, exist_ok=True)

    X, y = load_diabetes_csv(cfg.csv_path, cfg.target_col, cfg.positive_label)

    all_round_histories = []
    all_client_histories = []
    all_rdp_histories = []
    summary_rows = []

    for ablation in ["A1", "A2", "A3", "A4"]:
        for seed in cfg.seeds:
            print("\n" + "=" * 80)
            print(f"Running ablation {ablation} ({ABLATION_LABELS[ablation]}) | seed={seed}")
            print("=" * 80)

            seed_everything(seed)

            X_train, y_train, X_val, y_val, X_test, y_test, _ = split_data(X, y, cfg, seed)

            outputs = federated_train_pr_ashdp_personalised(
                X_train, y_train, X_val, y_val, X_test, y_test,
                cfg=cfg, seed=seed, ablation=ablation
            )

            round_df = outputs["round_history"].copy()
            client_df = outputs["client_round_history"].copy()
            rdp_df = outputs["rdp_round_history"].copy()

            all_round_histories.append(round_df)
            all_client_histories.append(client_df)
            if len(rdp_df) > 0:
                all_rdp_histories.append(rdp_df)

            summary_row = {
                "ablation": ablation,
                "ablation_name": ABLATION_LABELS[ablation],
                "seed": seed,
                "best_round": outputs["best_round"],
                "best_threshold": outputs["best_threshold"],
                "val_accuracy": outputs["final_val_metrics"]["accuracy"],
                "val_precision": outputs["final_val_metrics"]["precision"],
                "val_recall": outputs["final_val_metrics"]["recall"],
                "val_f1": outputs["final_val_metrics"]["f1"],
                "val_auprc": outputs["final_val_metrics"]["auprc"],
                "val_auroc": outputs["final_val_metrics"]["auroc"],
                "test_accuracy": outputs["final_test_metrics"]["accuracy"],
                "test_precision": outputs["final_test_metrics"]["precision"],
                "test_recall": outputs["final_test_metrics"]["recall"],
                "test_f1": outputs["final_test_metrics"]["f1"],
                "test_auprc": outputs["final_test_metrics"]["auprc"],
                "test_auroc": outputs["final_test_metrics"]["auroc"],
                "final_rdp_epsilon": outputs["final_privacy"].get("epsilon", np.nan),
                "final_rdp_best_alpha": outputs["final_privacy"].get("best_alpha", np.nan),
            }
            summary_rows.append(summary_row)

            seed_dir = os.path.join(cfg.output_dir, f"{ablation}_seed_{seed}")
            os.makedirs(seed_dir, exist_ok=True)

            round_df.to_csv(os.path.join(seed_dir, "round_history.csv"), index=False)
            client_df.to_csv(os.path.join(seed_dir, "client_round_history.csv"), index=False)
            rdp_df.to_csv(os.path.join(seed_dir, "rdp_round_history.csv"), index=False)

            torch.save(
                {
                    "global_encoder_state_dict": outputs["global_encoder"].state_dict(),
                    "global_head_state_dict": outputs["global_head"].state_dict(),
                    "client_heads_state_dict": {
                        cid: outputs["client_heads"][cid].state_dict()
                        for cid in outputs["client_heads"]
                    },
                    "config": asdict(cfg),
                    "ablation": ablation,
                    "ablation_name": ABLATION_LABELS[ablation],
                    "seed": seed,
                    "best_round": outputs["best_round"],
                    "best_threshold": outputs["best_threshold"],
                    "final_val_metrics": outputs["final_val_metrics"],
                    "final_test_metrics": outputs["final_test_metrics"],
                    "final_privacy": outputs["final_privacy"],
                },
                os.path.join(seed_dir, "best_model.pt"),
            )

    all_round_history_df = pd.concat(all_round_histories, ignore_index=True) if all_round_histories else pd.DataFrame()
    all_client_history_df = pd.concat(all_client_histories, ignore_index=True) if all_client_histories else pd.DataFrame()
    all_rdp_history_df = pd.concat(all_rdp_histories, ignore_index=True) if all_rdp_histories else pd.DataFrame()
    summary_df = pd.DataFrame(summary_rows)

    agg_rows = []
    group_cols = ["ablation", "ablation_name"]
    for keys, g in summary_df.groupby(group_cols):
        ablation, ablation_name = keys
        row = {"ablation": ablation, "ablation_name": ablation_name}
        for metric in [
            "test_accuracy", "test_precision", "test_recall", "test_f1", "test_auprc", "test_auroc",
            "val_accuracy", "val_precision", "val_recall", "val_f1", "val_auprc", "val_auroc",
            "final_rdp_epsilon"
        ]:
            mean_, std_ = summarize_metric(g[metric])
            row[f"{metric}_mean"] = mean_
            row[f"{metric}_std"] = std_
        agg_rows.append(row)

    ablation_comparison_df = pd.DataFrame(agg_rows).sort_values("ablation")

    all_round_history_df.to_csv(os.path.join(cfg.output_dir, "all_round_history.csv"), index=False)
    all_client_history_df.to_csv(os.path.join(cfg.output_dir, "all_client_round_history.csv"), index=False)
    all_rdp_history_df.to_csv(os.path.join(cfg.output_dir, "all_rdp_round_history.csv"), index=False)
    summary_df.to_csv(os.path.join(cfg.output_dir, "seed_level_summary.csv"), index=False)
    ablation_comparison_df.to_csv(os.path.join(cfg.output_dir, "ablation_comparison.csv"), index=False)

    with open(os.path.join(cfg.output_dir, "config.json"), "w") as f:
        json.dump(asdict(cfg), f, indent=2)

    print("\n================ ABLATION COMPARISON ================")
    print(ablation_comparison_df)

    return {
        "all_round_history": all_round_history_df,
        "all_client_history": all_client_history_df,
        "all_rdp_history": all_rdp_history_df,
        "seed_level_summary": summary_df,
        "ablation_comparison": ablation_comparison_df,
    }


# ============================================================
# 15. Entry Point
# ============================================================

def main():
    print("DEVICE:", DEVICE)
    print("\nConfiguration:")
    for k, v in asdict(CFG).items():
        print(f"{k}: {v}")

    outputs = run_all_ablation_experiments(CFG)

    print("\nSaved outputs to:", CFG.output_dir)
    print("\nSeed-level summary tail:")
    print(outputs["seed_level_summary"].tail())

    print("\nAblation comparison:")
    print(outputs["ablation_comparison"])

    return outputs


if __name__ == "__main__":
    outputs = main()
